# Convolutional Neural Network

### Importing the libraries

In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

## Part 1 - Data Preprocessing

### Preprocessing the Training set

In [2]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

training_set = train_datagen.flow_from_directory('dataset/train_data',
                                                 target_size = (128,128),
                                                 batch_size = 32,
                                                 class_mode = 'categorical')

Found 484 images belonging to 4 classes.


### Preprocessing the Test set

In [3]:
test_datagen = ImageDataGenerator(rescale = 1./255)
test_set = test_datagen.flow_from_directory('dataset/test_data',
                                            target_size = (128,128),
                                            batch_size = 32,
                                            class_mode = 'categorical')

Found 246 images belonging to 4 classes.


## Part 2 - Building the CNN

### Initialising the CNN

In [4]:
cnn = tf.keras.models.Sequential()

### Step 1 - Convolution

In [5]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu', input_shape=[128, 128, 3]))

#normalisation
cnn.add(tf.keras.layers.BatchNormalization())

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


### Step 2 - Pooling

In [6]:
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

### Adding a second convolutional layer

In [7]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'))
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

### Step 3 - Flattening

In [8]:
cnn.add(tf.keras.layers.Flatten())

### Step 4 - Full Connection

In [9]:
cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))

### Step 5 - Output Layer

In [10]:
cnn.add(tf.keras.layers.Dense(units=4, activation='softmax'))

## Part 3 - Training the CNN

### Compiling the CNN

In [11]:
cnn.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics = ['accuracy'])

### Training the CNN on the Training set and evaluating it on the Test set

In [12]:
cnn.fit(x = training_set, validation_data = test_set, epochs = 30)

Epoch 1/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 3s 138ms/step - accuracy: 0.4959 - loss: 3.3550 - val_accuracy: 0.8211 - val_loss: 1.2525
Epoch 2/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 130ms/step - accuracy: 0.6219 - loss: 1.0306 - val_accuracy: 0.8130 - val_loss: 1.1379
Epoch 3/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 130ms/step - accuracy: 0.6322 - loss: 0.9508 - val_accuracy: 0.8089 - val_loss: 1.2468
Epoch 4/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 147ms/step - accuracy: 0.6508 - loss: 0.8907 - val_accuracy: 0.8089 - val_loss: 1.1284
Epoch 5/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 126ms/step - accuracy: 0.6694 - loss: 0.8410 - val_accuracy: 0.8089 - val_loss: 1.1627
Epoch 6/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 125ms/step - accuracy: 0.6715 - loss: 0.8190 - val_accuracy: 0.8008 - val_loss: 1.1138
Epoch 7/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 128ms/step - accuracy: 0.6674 - loss: 0.8411 - val_accuracy: 0.8089 - val_loss: 0.9525
Epoch 8/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 137ms/step - accuracy: 0.6694 - loss: 0.8309 - val_accuracy: 0.

## Part 4 - Making a single prediction

In [23]:
import numpy as np
from tensorflow.keras.preprocessing import image

# test_image = image.load_img('dataset/single/r_rn.jpg', target_size = (128, 128))

# List of 3 images to test
image_paths = [
    'dataset/single/r_rn.jpg',
    'dataset/single/r_h.jpg',
    'dataset/single/r_f.jpg'
]

# test_image = image.img_to_array(test_image)/ 255.0
# test_image = np.expand_dims(test_image, axis = 0)

# Load and preprocess all 3 images
test_images = []
for path in image_paths:
    img = image.load_img(path, target_size=(128, 128))
    img_array = image.img_to_array(img) / 255.0
    test_images.append(img_array)


    
# Convert list to numpy array (batch of 3)
test_images = np.array(test_images)

# Predict
results = cnn.predict(test_images)

# result = cnn.predict(test_image)

# Get class mapping
class_indices = training_set.class_indices
print("Class indices:", class_indices)

# Convert prediction to readable label
predicted_indices = np.argmax(results, axis=1)


# prediction = list(class_indices.keys())[predicted_class]

# Convert predictions to readable labels
predicted_labels = [list(class_indices.keys())[i] for i in predicted_indices]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
Class indices: {'fever': 0, 'healthy': 1, 'running nose': 2, 'sore throat': 3}


In [24]:
# print(f"Predicted class: {prediction}")

# Display results neatly
for i, path in enumerate(image_paths):
    print(f"{path} → Predicted: {predicted_labels[i]} (Confidence: {np.max(results[i]):.2f})")


dataset/single/r_rn.jpg → Predicted: running nose (Confidence: 0.66)
dataset/single/r_h.jpg → Predicted: healthy (Confidence: 0.97)
dataset/single/r_f.jpg → Predicted: healthy (Confidence: 0.93)
